```mermaid
flowchart LR
    A0["00"] --> A1a["01a"] --> A1b["01b"] --> A2["02"] --> A3["03"] --> A4a["04a"] --> A4b["04b"]
    A4b --> A5a["05a"] --> A5b["05b"] --> A6a["06a"] --> A6b["06b"]
    A6b --> A7["07"] --> A8a["08a"] --> A8b["08b"]
    A8b --> A9["09"] --> A10["10"] --> A11["11"] --> A12["12"] 
    
    classDef normal fill:#f8f9fa,stroke:#adb5bd,stroke-width:1px,color:#111;
    classDef done fill:#e8f7f0,stroke:#198754,stroke-width:1.5px,color:#111;
    classDef current fill:#fff3cd,stroke:#ff8c00,stroke-width:2px,color:#111;
    
    class A0,A1a,A1b,A2,A3,A4a,A4b,A5a,A5b done;
    class A6a current;
    class A6b,A7,A8a,A8b,A9,A10,A11,A12 normal;
```

# Notebook 06a — Named Entity Recognition (NER)

This notebook introduces Named Entity Recognition (NER) as a way to detect and analyze references to people, places, organizations, works, schools, and concepts in philosophical texts.

The focus of this notebook is:
- using pretrained spaCy NER models
- understanding domain mismatch and annotation limitations
- exploring entity distributions and co-occurrence patterns over time
- preparing the corpus for possible domain adaptation or custom NER training later

Because this course runs on CPU-friendly student laptops, we use a staged workflow:
1. explore pretrained NER pipelines
2. evaluate their usefulness on philosophical texts
3. optionally prepare lightweight custom training data
4. optionally compare with transformer-based models (inference only)

We do NOT fine-tune a transformer model in class. If desired, the instructor can later train a model once on GPU and distribute the trained pipeline for inference and evaluation.

## Why pretrained spaCy NER first?

Pretrained NER models are fast, lightweight, and easy to inspect. They are ideal for understanding:
- what NER systems detect well
- what they miss in specialized corpora
- how annotation decisions shape downstream analysis

Philosophical texts are challenging because:
- many concepts are abstract rather than named entities
- works and schools are inconsistently formatted
- older language differs from the data used to train modern NER models

This makes domain mismatch an important methodological issue.

In [ ]:
# -----------------------------
# PATHS
# -----------------------------
from __future__ import annotations

from pathlib import Path
from collections import Counter
import re

import numpy as np
import pandas as pd

import spacy
from spacy.tokens import DocBin

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# -----------------------------
# PATHS
# -----------------------------
PROJECT_ROOT = Path('.')

DATA_DIR = PROJECT_ROOT / 'data'
META_DIR = PROJECT_ROOT / 'analysis' / 'tables'
PROCESSED_DIR = DATA_DIR / 'processed'

TEXTS_DIR = PROCESSED_DIR / 'gutenberg' / 'texts_cleaned'

ANALYSIS_DIR = PROJECT_ROOT / 'analysis'
FIGURES_DIR = ANALYSIS_DIR / 'figures'
TABLES_DIR = ANALYSIS_DIR / 'tables'
REPORTS_DIR = ANALYSIS_DIR / 'reports'

CACHE_DIR = PROJECT_ROOT / 'cache'

# Previous notebook artifacts
DOC_INDEX = META_DIR / 'nb03-doc_index.csv'

SPACY_MODEL = 'en_core_web_sm'

print('DOC_INDEX:', DOC_INDEX)

In [ ]:
# Helper to extract start year from time_bin string and introduce a chronological order
def get_start_year(b: str) -> int:
    try:
        return int(b.split('–')[0] if '–' in b else b.split('-')[0])
    except (ValueError, IndexError, AttributeError):
        return 0

## Load the corpus index

In [ ]:
# Test on a small sample
MAX_DOCS = None

df = pd.read_csv(DOC_INDEX)

# Convert years from float to int
df["publication_year"] = pd.to_numeric(df["publication_year"], errors="coerce").astype("Int64")

if MAX_DOCS is not None:
    df = df.head(MAX_DOCS).copy()

print(f"\nLoaded metadata for {len(df)} documents' chunks.")
display(df.head())

## Load spaCy pipeline

We use a pretrained English spaCy pipeline. This notebook focuses on inference and exploration, not heavy training.

In [ ]:
print('Loading spaCy model:', SPACY_MODEL)
nlp = spacy.load(SPACY_MODEL)
print('Pipeline:', nlp.pipe_names)

## Load annotated documents from Notebook 05a

Notebook 05a serialized the corpus as a spaCy `DocBin` so we do not need to rerun annotation.

In [ ]:
# Path to .spacy split files
SPLIT_DIR = PROCESSED_DIR / 'nb05-corpus-split'

# Get all .spacy files in the directory
spacy_files = sorted(SPLIT_DIR.glob("*.spacy"))
print(f"\nFound {len(spacy_files)} split files.")

# Load all docs into a single list
all_docs = []
for filepath in spacy_files:
    docbin = DocBin().from_disk(filepath)
    docs = list(docbin.get_docs(nlp.vocab))
    all_docs.extend(docs)

print(f"\nLoaded {len(all_docs)} documents (chunks) total.")

## Quick NER inspection

We inspect entity spans predicted by the pretrained model on a sample document.

In [ ]:
sample_doc = docs[0]

rows = []
for ent in sample_doc.ents:
    rows.append({
        'text': ent.text,
        'label': ent.label_,
        'start': ent.start_char,
        'end': ent.end_char,
    })

ent_df = pd.DataFrame(rows)

print('Entities detected:', len(ent_df))
display(ent_df.head(20))

## Entity labels

spaCy predicts labels such as:
- PERSON
- ORG
- GPE
- WORK_OF_ART
- DATE

A useful exercise is to inspect which labels work well on philosophical texts and which fail because of domain mismatch.

In [ ]:
label_counter = Counter()

for doc in docs:
    label_counter.update(ent.label_ for ent in doc.ents)

label_df = (
    pd.DataFrame(label_counter.items(), columns=['label', 'count'])
      .sort_values('count', ascending=False)
)

display(label_df)

## Visualize entity label distribution

In [ ]:
# Sort labels by count (descending) for better readability
label_df_sorted = label_df.sort_values('count', ascending=True)   # ascending for horizontal

plt.figure(figsize=(10, 6))
sns.barplot(data=label_df_sorted, x='count', y='label', color='teal')
plt.title('Entity label distribution')
plt.xlabel('Count')
plt.ylabel('Entity label')
plt.tight_layout()
plt.show()

## Aggregate entity frequencies

We now aggregate entity mentions across the corpus. This is useful for identifying influential philosophers, recurring works, locations, and institutions.

# Fill the gap

In [ ]:
# =============================================== YOUR CODE HERE ===============================================
# Explore the entities stored in `doc.ents` 
# You will be able to add the entities that should be excluded in in the set below


In [ ]:
# For a single Doc
# persons = [ent for ent in doc.ents if ent.label_ == "PERSON"]

# For all chunks (if you have a list of docs)
all_persons = []
for doc in docs:
    all_persons.extend([ent for ent in doc.ents if ent.label_ == "PERSON"])

In [ ]:
persons = list(set(all_persons))

In [ ]:
persons[20:40]

In [ ]:
def is_valid_entity(ent: str, excluded_entities: set, excluded_labels: set):
    """Return True if entity should be kept."""
    # Exclude if text is in excluded_entities (lowercase comparison)
    if ent.text.lower() in excluded_entities:
        return False
    
    # Exclude if the entity text is purely digits
    if ent.text.isdigit():
        return False
    
    # Exclude if label is in the excluded list
    if ent.label_ in excluded_labels:
        return False
    
    # Optionally exclude very short entities (e.g., single characters)
    if len(ent.text.strip()) < 2:
        return False
    
    return True

In [ ]:
# =============================================== YOUR CODE HERE ===============================================
# Define a custom set of named entities to exclude
EXCLUDED_ENTITIES = {"XV",
                     "xv",
                     "xxxv"
}

# Labels you might want to exclude (common noise in historical texts)
EXCLUDED_LABELS = {
    "CARDINAL", "ORDINAL"
}

In [ ]:
# Now extract entities with filtering
entity_rows = []

for doc in all_docs:  # all_docs is the list of loaded Doc objects
    for ent in doc.ents:
        if is_valid_entity(ent, EXCLUDED_ENTITIES, EXCLUDED_LABELS):
            entity_rows.append({
                'pg_id': doc.user_data.get('pg_id'),
                'title': doc.user_data.get('title'),
                'time_bin': doc.user_data.get('time_bin'),
                'entity': ent.text,
                'label': ent.label_,
            })

entities = pd.DataFrame(entity_rows)

print('\nEntity mentions (after filtering):', len(entities), '\n')
display(entities.head())

In [ ]:
# # Aggregate entities without filtering

# entity_rows = []

# for doc in docs:
#     for ent in doc.ents:
#         entity_rows.append({
#             'pg_id': doc.user_data.get('pg_id'),
#             'filename': doc.user_data.get('filename'),
#             'title': doc.user_data.get('title'),
#             'time_bin': doc.user_data.get('time_bin'),
#             'entity': ent.text,
#             'label': ent.label_,
#         })

# entities = pd.DataFrame(entity_rows)

# print('Entity mentions:', len(entities))
# display(entities.head())

In [ ]:
top_entities = (
    entities.groupby(['label', 'entity'])
            .size()
            .reset_index(name='count')
            .sort_values('count', ascending=False)
)

display(top_entities.head(30))

## Entity frequencies over time

This section explores how entity mentions vary across historical periods.

In [ ]:
# Group and aggregate
entity_time = (
    entities.groupby(['time_bin', 'label'])
            .size()
            .reset_index(name='count')
)

# Sort chronologically by start year
entity_time['start_year'] = entity_time['time_bin'].apply(get_start_year)
entity_time = entity_time.sort_values('start_year').drop(columns=['start_year'])

# Plot
plt.figure(figsize=(12,6))
sns.lineplot(data=entity_time, x='time_bin', y='count', hue='label', marker='o')
plt.xticks(rotation=45)
plt.title('Entity mentions over time')
plt.tight_layout()
plt.show()

## Entity co-occurrence exploration

Two entities are considered to co-occur if they appear in the same document. This provides a lightweight network view of the corpus.

In [ ]:
# With unfiltered entities

from itertools import combinations

co_rows = []

for doc in docs:
    ents = sorted(set(ent.text for ent in doc.ents if len(ent.text) > 2))

    for a, b in combinations(ents, 2):
        co_rows.append((a, b))

co_df = pd.DataFrame(co_rows, columns=['entity_a', 'entity_b'])

co_counts = (
    co_df.groupby(['entity_a', 'entity_b'])
         .size()
         .reset_index(name='count')
         .sort_values('count', ascending=False)
)

display(co_counts.head(30))

In [ ]:
# With FILTERED entities
# Group filtered entity texts by original document
from collections import defaultdict
doc_entities_dict = defaultdict(set)

for doc in docs:
    pg_id = doc.user_data.get('pg_id')
    if pg_id is None:
        continue
    for ent in doc.ents:
        if is_valid_entity(ent, EXCLUDED_ENTITIES, EXCLUDED_LABELS):
            doc_entities_dict[pg_id].add(ent.text)

# Generate combinations per original document
co_rows = []
for pg_id, ent_set in doc_entities_dict.items():
    ents = sorted(ent_set)
    for a, b in combinations(ents, 2):
        co_rows.append((a, b))

co_df = pd.DataFrame(co_rows, columns=['entity_a', 'entity_b'])

co_counts = (
    co_df.groupby(['entity_a', 'entity_b'])
         .size()
         .reset_index(name='count')
         .sort_values('count', ascending=False)
)

display(co_counts.head(30))

### Entity co‑occurrence matrix (from filtered entities)
Build a document‑entity matrix and compute pairwise co‑occurrence counts
using sparse matrix multiplication, which is faster than Python loops.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from scipy.sparse import triu

# ------------------------------------------------------------------------------
# 1. (Optional) Keep only entities that appear in at least `min_doc_freq` documents
#    This reduces matrix size and removes very rare entities.
# ------------------------------------------------------------------------------
min_doc_freq = 3   # adjust as needed
entity_doc_counts = entities.groupby('entity')['pg_id'].nunique()
keep_entities = entity_doc_counts[entity_doc_counts >= min_doc_freq].index
entities_filtered = entities[entities['entity'].isin(keep_entities)]

print(f"Entities after min_doc_freq={min_doc_freq}: {len(keep_entities)} unique")
print(f"Mentions remaining: {len(entities_filtered)}")

# ------------------------------------------------------------------------------
# 2. Build document‑entity strings (one string per pg_id)
#    Replace spaces with underscores so that multi‑word entities become a single token.
# ------------------------------------------------------------------------------
doc_entity_list = (
    entities_filtered.groupby('pg_id')['entity']
    .apply(lambda x: ' '.join(set(ent.replace(' ', '_') for ent in x)))
)

print(f"Number of documents (pg_id) with entities: {len(doc_entity_list)}")

# ------------------------------------------------------------------------------
# 3. Vectorize using CountVectorizer with binary=True
#    The default token pattern `(?u)\b\w+\b` treats underscores as word characters,
#    so `New_York` becomes a single token.
# ------------------------------------------------------------------------------
vectorizer = CountVectorizer(
    token_pattern=r'(?u)\b\w+\b',   # underscores are part of \w
    binary=True                     # 1 if entity appears in document, 0 otherwise
)

X = vectorizer.fit_transform(doc_entity_list)
print(f"Document‑entity matrix shape: {X.shape}")

# ------------------------------------------------------------------------------
# 4. Compute co‑occurrence matrix: X^T * X
# ------------------------------------------------------------------------------
co_occurrence = X.T @ X
print(f"Co‑occurrence matrix shape: {co_occurrence.shape}")

# ------------------------------------------------------------------------------
# 5. Extract upper triangle (k=1) to avoid duplicates and self‑pairs
# ------------------------------------------------------------------------------
co_occ_triu = triu(co_occurrence, k=1)
coo = co_occ_triu.tocoo()

# Get entity labels (with underscores)
entity_labels = vectorizer.get_feature_names_out()

# Build DataFrame
co_counts_m = pd.DataFrame({
    'entity_a': entity_labels[coo.row],
    'entity_b': entity_labels[coo.col],
    'count': coo.data
})

# ------------------------------------------------------------------------------
# 6. Sort and display
# ------------------------------------------------------------------------------
co_counts_m = co_counts_m.sort_values('count', ascending=False).reset_index(drop=True)

# (Optional) Restore original entity text by replacing underscores with spaces
co_counts_m['entity_a'] = co_counts_m['entity_a'].str.replace('_', ' ')
co_counts_m['entity_b'] = co_counts_m['entity_b'].str.replace('_', ' ')

print(f"\nTotal unique entity pairs: {len(co_counts)}\n")
print("Top 30 co‑occurring entity pairs:")
display(co_counts_m.head(30))

# ------------------------------------------------------------------------------
# 7. Save if needed
# ------------------------------------------------------------------------------
# co_counts_m.to_csv(OUTPUT_DIR / 'tables' / 'nb06-entity_cooccurrence_matrix.csv', index=False)

## Rule-based augmentation with EntityRuler

Pretrained models often miss domain-specific concepts. A lightweight solution is to add rule-based patterns with spaCy's `EntityRuler`.

Examples:
- philosopher names
- philosophical schools
- canonical works
- abstract concepts

This provides a bridge toward custom NER without expensive training.

In [ ]:
# Example: lightweight rule-based extension

patterns = [
    {'label': 'PHILOSOPHER', 'pattern': 'Aristotle'},
    {'label': 'PHILOSOPHER', 'pattern': 'Kant'},
    {'label': 'PHILOSOPHER', 'pattern': 'Nietzsche'},
    {'label': 'SCHOOL', 'pattern': 'Stoicism'},
    {'label': 'SCHOOL', 'pattern': 'Empiricism'},
]

# Build a fresh pipeline copy to avoid mutating the original
nlp_rules = spacy.load(SPACY_MODEL)

# Match phrase patterns case-insensitively
ruler = nlp_rules.add_pipe(
    'entity_ruler',
    before='ner',
    config={'phrase_matcher_attr': 'LOWER'}
)
ruler.add_patterns(patterns)

example = nlp_rules('Kant criticizes empiricism and discusses Aristotle.')

[(ent.text, ent.label_) for ent in example.ents]

## Saving outputs for later notebooks

In [ ]:
entities.to_csv(TABLES_DIR / 'nb06-entities.csv', index=False)
label_df.to_csv(TABLES_DIR / 'nb06-entity_label_distribution.csv', index=False)
co_counts.to_csv(TABLES_DIR / 'nb06-entity_cooccurrence.csv', index=False)
co_counts_m.to_csv(TABLES_DIR / 'nb06-entity_cooccurrence-matrix.csv', index=False)

print('Saved entity tables to:', TABLES_DIR)

## Reflection questions

- Which entity types are easiest for the pretrained model?
- Which entities are systematically missed?
- How does historical/philosophical language affect NER quality?
- Which kinds of entities are most useful for studying knowledge dynamics?
- When is a rule-based system preferable to full model training?

# Conclusion and transition to Notebook 06b

In this notebook, we used a pretrained NER system to explore what kinds of entities spaCy can detect in philosophical texts. This gave us a useful first inventory of people, places, works, dates, and other named entities across the corpus. At the same time, it also revealed an important methodological limitation: pretrained NER reflects the categories and training data of a general-purpose model, not the specific research interests of a given investigation.

For philosophical texts, this means that some detected entities are genuinely useful, while others are noisy, misleading, or simply irrelevant concepts. It also means that many important philosophical ideas are not recognized at all, because they do not belong to standard named-entity categories such as `PERSON`, `GPE`, or `ORG`.

This is the point where we move from **automatic detection** to **research design**.

In **Notebook 06b**, we will build a more selective and interpretive NER workflow. Instead of treating all detected entities as equally relevant, we will review the entity lists generated from this notebook and decide which ones are worth tracking. We will also create our own list of philosophical concepts and add them through spaCy’s `EntityRuler`, which allows us to define a rule-based custom NER layer.

### Homework for Notebook 06b

Before or during Notebook 06b, you will:

1. open the generated entity selection file(s)
2. review the detected named entities grouped by label
3. change `selected` from `False` to `True` for the entities you want to keep
4. leave `False` for entities you want to exclude
5. open `analysis/nb06-custom_concepts.txt`
6. replace the example entries with philosophical concepts you want to track
7. rerun the notebook analysis with:
   - your selected named entities
   - your custom philosophical concepts added through `EntityRuler`

The goal is to produce a **custom NER tracking analysis** that reflects your own interpretive choices. Different students may choose different entities and concepts, and that is part of the exercise: the final visualizations will help you see how analytical results change when you redefine what counts as important in the corpus.

> **Key idea:** NER is not only a model output. It is also a methodological choice about what kinds of references and concepts matter for your research question.



```mermaid
flowchart TB
    A0["00<br/>Bootcamp"] --> P1

    subgraph P1["Part I — Corpus building and analysis"]
        direction LR
        A1a["01a<br/>Corpus metadata"] --> A1b["01b<br/>Corpus building"] --> A2["02<br/>Preprocessing"] --> A3["03<br/>Distributions + time"] --> A4a["04a<br/>Lexical exploration"] --> A4b["04b<br/>Embedding"]
    end

    subgraph P2["Part II — Linguistic annotations"]
        direction LR
        A5a["05a<br/>spaCy annotation"] --> A5b["05b<br/>Relation extraction"] --> A6a["06a<br/>NER"] --> A6b["06b<br/>Custom NER"]
    end

    subgraph P3["Part III — Representations"]
        direction LR
        A7["07<br/>BoW + TF-IDF"] --> A8a["08a<br/>Embeddings"] --> A8b["08b<br/>Transformers"]
    end

    subgraph P4["Part IV — Models and interpretation"]
        direction LR
        A9["09<br/>Classification"] --> A10["10<br/>Custom NER training"] --> A11["11<br/>Topic modeling"] --> A12["12<br/>Semantic shift"]
    end

    P1 --> P2
    P2 --> P3
    P3 --> P4

    classDef start fill:#f3f0ff,stroke:#6f42c1,stroke-width:1.5px,color:#111;
    classDef prep fill:#eef7ff,stroke:#1f77b4,stroke-width:1.5px,color:#111;
    classDef annot fill:#eefaf0,stroke:#2ca02c,stroke-width:1.5px,color:#111;
    classDef repr fill:#fff7e6,stroke:#ff8c00,stroke-width:1.5px,color:#111;
    classDef model fill:#fff0f0,stroke:#d62728,stroke-width:1.5px,color:#111;

    classDef highlight fill:#fff3b0,stroke:#f5a623,stroke-width:4px,color:#111;

    class A1a,A1b,A2,A3,A4a,A4b prep;
    class A5a,A5b,A6a,A6b annot;
    class A7,A8a,A8b repr;
    class A9,A10,A11,A12 model;

    class A6a highlight;
```